In [0]:
#/Volumes/workspace/data/data_joins/dataset/

In [0]:

dbutils.fs.ls("/Volumes/workspace/data/data_joins/dataset/")



[FileInfo(path='dbfs:/Volumes/workspace/data/data_joins/dataset/customers_medium.csv', name='customers_medium.csv', size=38728, modificationTime=1778027425000),
 FileInfo(path='dbfs:/Volumes/workspace/data/data_joins/dataset/menu_items.csv', name='menu_items.csv', size=6725, modificationTime=1778027425000),
 FileInfo(path='dbfs:/Volumes/workspace/data/data_joins/dataset/order_items (2).csv', name='order_items (2).csv', size=257134, modificationTime=1778027425000),
 FileInfo(path='dbfs:/Volumes/workspace/data/data_joins/dataset/orders_medium.csv', name='orders_medium.csv', size=286712, modificationTime=1778027425000),
 FileInfo(path='dbfs:/Volumes/workspace/data/data_joins/dataset/restaurants.csv', name='restaurants.csv', size=3037, modificationTime=1778027425000)]

In [0]:
df_files = spark.createDataFrame(
    dbutils.fs.ls("/Volumes/workspace/data/data_joins/dataset/")
    )
display(df_files)

path,name,size,modificationTime
dbfs:/Volumes/workspace/data/data_joins/dataset/customers_medium.csv,customers_medium.csv,38728,1778027425000
dbfs:/Volumes/workspace/data/data_joins/dataset/menu_items.csv,menu_items.csv,6725,1778027425000
dbfs:/Volumes/workspace/data/data_joins/dataset/order_items (2).csv,order_items (2).csv,257134,1778027425000
dbfs:/Volumes/workspace/data/data_joins/dataset/orders_medium.csv,orders_medium.csv,286712,1778027425000
dbfs:/Volumes/workspace/data/data_joins/dataset/restaurants.csv,restaurants.csv,3037,1778027425000


In [0]:
from pyspark.sql.functions import col, broadcast
base_path="/Volumes/workspace/data/data_joins/dataset/"

In [0]:
customers=spark.read.csv(base_path+"customers_medium.csv",header=True,inferSchema=True)
menu=spark.read.csv(base_path+"menu_item.csv",header=True,inferSchema=True)
orders=spark.read.csv(base_path+"order_items (2).csv",header=True,inferSchema=True)
order_items =spark.read.csv(base_path+"orders_medium.csv",header=True,inferSchema=True)
restaurants =spark.read.csv(base_path+"restaurants.csv",header=True,inferSchema=True)


In [0]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [0]:
#find the all customers who have placed orders

inner_df=customers.join(orderitem,"customer_id","inner")


In [0]:
display(inner_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7550510766131361>, line 1
----> 1 display(inner_df)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks support"
     95       ) from e
---> 96 if df.isStreaming:
     97     self.cf_helper.display_streaming_dataframe(df, config, 

In [0]:
#find all customers and their orders (including those who have never ordered)
#

In [0]:
orderitem.printSchema()
ordermedium.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- restaurant_id: string (nullable = true)
 |-- order_time: date (nullable = true)
 |-- delivery_time: timestamp (nullable = true)
 |-- status: string (nullable = true)



In [0]:
#find the orders that have. no items
missing_items_df=orders.join(order_items,"order_id","left") \
    .filter(col("item_id").isNull())

In [0]:
display(missing_items_df)

order_id,item_id,quantity,price,customer_id,restaurant_id,order_time,delivery_time,status


In [0]:
final_df=orders \
    .join(customers,"customer_id") \
    .join(order_items,"order_id") 
display(final_df)


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5688072202719437>, line 4
      1 final_df=orders \
      2     .join(customers,"customer_id") \
      3     .join(order_items,"order_id") 
----> 4 display(final_df)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks support"
   

In [0]:
#broadcast join -optimize join between large and small tables 
from pyspark.sql.functions import broadcast
 
optimize_df = orders.join (broadcast(customers), "customer_id")
 
display(optimize_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5688072202719438>, line 6
      2 from pyspark.sql.functions import broadcast
      4 optimize_df = orders.join (broadcast(customers), "customer_id")
----> 6 display(optimize_df)

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:96, in Display.display_connect_table(self, df, **kwargs)
     91 except Exception as e:
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks